In [8]:
import pandas as pd
import numpy as np
import math
df = pd.read_csv("adult.csv")

# BELOW: attempt at binning, but it lowered F1 score so commented out

#df['Age_Category'] = pd.cut(df['age'], bins=4, labels=['Young Adult', 'Adult', 'Middle Age', 'Older'])
#df['capital-gain_category'] = pd.cut(df['capital-gain'], bins=4, labels=['Small', 'Medium', 'Large', 'Extra Large'])
#df['capital-loss_category'] = pd.cut(df['capital-loss'], bins=4, labels=['Small', 'Medium', 'Large', 'Extra Large'])
#df['hours-per-week_category'] = pd.cut(df['hours-per-week'], bins=3, labels=['below average', 'average', 'above average'])
#df = df.drop(columns=["fnlwgt", 'age', 'capital-gain', 'capital-loss', 'hours-per-week', "educational-num"])

df = df.drop(columns =["fnlwgt", "educational-num"]) #dropping unhelpful features
#shuffles data for the sake of randomness and representation
#using a random_state to ensure reproducibility
df = df.sample(n=10000, random_state=42).reset_index(drop=True) 
df["income"] = df["income"].str.strip()
length = len(df)
train_length = int(length* 0.75)
train = df.iloc[:train_length]
test = df.iloc[train_length:]

def entropy(y_vector) : #caclulate entropy for individual node 
    l50 = len((y_vector[y_vector == "<=50K"]))/len(y_vector)
    g50 = len((y_vector[y_vector == ">50K"]))/len(y_vector)
    probs = [p for p in [l50, g50] if p > 0]
    return -sum(p * math.log2(p) for p in probs)

def information_gain (df) : #calculate the IG between parent and its children
    parent = entropy(df["income"]) 
    max_ent = "" #column with the max entropy value
    max_ent_value = -1 #max entropy value, default is -1 because it could never be -1
    for col in df.columns : #cycle through each column 
        if col == "income" :
            continue
        weighted_ent = [] #list of individual value's entropy
        values = df[col].unique()
        for val in values: #cycle through the unique values found in the column calculating for each
            subset = df[df[col] == val] 
            weighted_ent.append((len(subset)/len(df)) * entropy(subset["income"]))
        weighted_ent_total = sum(weighted_ent)
        ig = parent -weighted_ent_total
        if(ig > max_ent_value) :
            max_ent_value = ig
            max_ent = col
    return max_ent_value, max_ent


def build_tree (df, depth = 0, max_depth = 2) :
    if len(df["income"].unique()) == 1: #stopping condition: if the node is pure
        return df["income"].iloc[0]
    if len(df.columns) == 1: #stopping condition: if theres only one attribute left
        return df["income"].mode()[0]
    if depth >= max_depth:
        return df["income"].mode()[0]
    ig, col = information_gain(df)
    tree = {"attribute" : col, "split":{}} #placing output from information gain into tree
    for val in df[col].unique() : #cycling through unique values in col
        subset = df[df[col] == val]
        subset = subset.drop(columns = [col]) 
        tree["split"][val] = build_tree(subset, depth = depth + 1, max_depth = max_depth) #builds additional branch using smaller and smaller subsets
    return tree

tree = build_tree(train, max_depth = 10)


def predict_row(tree, row) :
    if isinstance(tree, str) :#if the current limb is a string that means that it is a leaf and theres a classification already
        return tree #return the classification instead of continuing
    attr = tree["attribute"] 
    val = row[attr] #looking at the same location in the row as the trees current attribute
    if val not in tree["split"] : #if the value is not found in the training split, go with the most commonly occuring class in the dataset
        return train["income"].mode()[0]
    return predict_row(tree["split"][val], row) #continue predicting by splitting down a subsection of the tree

y_actual = test["income"].tolist() 
y_pred = [predict_row(tree, test.iloc[i]) for i in range(len(test))] #getting prediction of y_actual
y_actual = np.array(y_actual)
y_pred = np.array(y_pred)
labels = [">50K", "<=50K"]

f1 = []
for label in labels:
    TP = sum((y_pred == label) & (y_actual == label))
    FP = sum((y_pred == label) & (y_actual != label))
    FN = sum((y_pred != label) & (y_actual == label))
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    F1 = 2 * ((precision * recall) / (precision + recall)) if (precision + recall) > 0 else 0
    f1.append(F1)




In [9]:
def boost_tree (df, n = 4) :
    #initializing weights for training
    #adaboosts updates weights iteratively to focus more on hard to classify samples
    my_list = [1/len(df)] * len(df)
    weights = np.array(my_list)
    trees = []
    importance = []
    
    for i in range(n) :
        correctness = [0]*len(df) #0 if incorrect 1 if correct
        z = []
        #sampling data using updated weights
        df2 = df.sample(n=len(df), replace = True, weights = weights)
        #train a weak learner
        tree = build_tree(df2, max_depth= 1) #change second parameter for depth of individual trees
        trees.append(tree)
        #actual and predicted labels from dataset and tree
        y_actual = df["income"].tolist()
        y_pred = [predict_row(tree, df.iloc[i]) for i in range(len(df))]
        y_actual = np.array(y_actual)
        y_pred = np.array(y_pred)
        #determining which were classified correctly 
        for j in range (len(df2)) :
            incorrect = y_pred[j] != y_actual[j]
            if incorrect :
                correctness[j] = 0
            if  not incorrect :
                correctness[j] = 1
        #computed weighted error
        error = (sum(weights* (1- np.array(correctness))))/sum(weights)
        alpha = 0.5 * math.log((1 - error) / (error + 1e-10)) #compute learning weight using adaboost formula
        alpha = max(min(alpha, 2.5), -2.5) #clipping to prevent extreme values

        importance.append(alpha)
        #calculating z (normalizer)
        for j in range (len(df2)) :
            if (correctness[j] == 0) : 
                z.append(weights[j] * math.e**(importance[i]))
            if(correctness[j] == 1) :
                z.append(weights[j] * math.e**(-importance[i]))
        z = sum(z)
        #calculating and updating the weights
        for j in range (len(df2)) :
            if (correctness[j] == 0) :
                weights[j] = ((weights[j])/z)*math.e**(alpha)
            if(correctness[j] == 1) : 
                weights[j] = ((weights[j])/z)*math.e**(-alpha)
        weights = weights / np.sum(weights) #normalizing weights
        print("Round: ", i + 1, "|", "weighted error: ", error, "|",  "alpha: ",  alpha)
    return trees, importance, correctness

def predict_boosted_tree(trees, importance, test) :
    predictions = []
    for i in range(len(test)) :
        scores = {">50K": 0, "<=50K": 0}
        for j in range(len(trees)):
            pred = predict_row(trees[j], test.iloc[i]) 
            weight = importance[j]
            scores[pred] = scores[pred] + weight
        final = max(scores, key = scores.get )
        predictions.append(final)
    return predictions

#BELOW: created for calculating f1 score of the ensemble before considering baseline models, commented out for efficiency
    
#trees, importance, correctness = boost_tree(train, 100)
#preds = predict_boosted_tree(trees, importance, test)
#y_actual = test["income"].tolist() 
#y_actual = np.array(y_actual)
#preds = np.array(preds)
#labels = ["<=50K", ">50K"]
    
#f1 = []
#for label in labels:
    #   TP = sum((preds == label) & (y_actual == label))
    #  FP = sum((preds == label) & (y_actual != label))
    # FN = sum((preds != label) & (y_actual == label))
    # precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    # recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    # F1 = 2 * ((precision * recall) / (precision + recall)) if (precision + recall) > 0 else 0
    #f1.append(F1)
    #print("F1 Score:", sum(f1)/len(labels)) 

In [10]:
#baseline models calculated individually
from sklearn.linear_model import Perceptron
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import pandas as pd
#numerical encoding required for sklearn's naive bayes and perceptron
x_train = pd.get_dummies(train.drop(columns=["income"]))
x_test = pd.get_dummies(test.drop(columns=["income"]))
x_train, x_test = x_train.align(x_test, join='left', axis=1, fill_value=0)

y_train = train["income"]
y_test = test["income"]

# Perceptron
perc = Perceptron(max_iter=1000, random_state=42)
perc.fit(x_train, y_train)
perc_pred = perc.predict(x_test)

# Naive Bayes
nb = GaussianNB()
nb.fit(x_train, y_train)
nb_pred = nb.predict(x_test)

# Random Forest
rf = RandomForestClassifier(random_state=42)
rf.fit(x_train, y_train)
rf_pred = rf.predict(x_test)

print("Perceptron F1:", f1_score(y_test, perc_pred, average='macro'))
print("Naive Bayes F1:", f1_score(y_test, nb_pred, average='macro'))
print("Random Forest F1:", f1_score(y_test, rf_pred, average='macro'))


Perceptron F1: 0.6634790125329966
Naive Bayes F1: 0.7577871912330432
Random Forest F1: 0.797194843185661


In [11]:
from scipy.stats import ttest_rel

def get_f1 () : 
    f1_all = [] #adaboost f1 score
    f1_2 = []
    f1_3 = []
    f1_4 = []
    for i in range(5):
        #reinitilizing each of the following, so the sample can be updated randomly each iteration
        f1 = []
        df = pd.read_csv("adult.csv")  
        df = df.drop(columns =["fnlwgt", "educational-num"])
        df = df.sample(n=10000, random_state = i).reset_index(drop=True) 
        df["income"] = df["income"].str.strip()
        length = len(df)
        train_length = int(length* 0.75)
        train = df.iloc[:train_length]
        test = df.iloc[train_length:]
        #training AdaBoost ensemble
        trees, importance, correctness = boost_tree(train, 50) #change the second parameter to change the amount of trees
        preds = predict_boosted_tree(trees, importance, test) #receiving prediction given the test data
        y_actual = test["income"].tolist() 
        y_actual = np.array(y_actual)
        preds = np.array(preds)
        labels = ["<=50K", ">50K"]
       #calculating f1 scores for each of the 4 models
        for label in labels:
            TP = sum((preds == label) & (y_actual == label))
            FP = sum((preds == label) & (y_actual != label))
            FN = sum((preds != label) & (y_actual == label))
            precision = TP / (TP + FP) if (TP + FP) > 0 else 0
            recall = TP / (TP + FN) if (TP + FN) > 0 else 0
            F1 = 2 * ((precision * recall) / (precision + recall)) if (precision + recall) > 0 else 0
            f1.append(F1)
        #numerical encoding required for sklearn's naive bayes and perceptron
        x_train = pd.get_dummies(train.drop(columns=["income"]))
        x_test = pd.get_dummies(test.drop(columns=["income"]))
        x_train, x_test = x_train.align(x_test, join='left', axis=1, fill_value=0)
    
        y_train = train["income"]
        y_test = test["income"]
        #each of the baseline models are using sklearn to retreive a prediction
        # Perceptron
        perc = Perceptron(max_iter=1000, random_state=42)
        perc.fit(x_train, y_train)
        perc_pred = perc.predict(x_test)
        
        # Naive Bayes
        nb = GaussianNB()
        nb.fit(x_train, y_train)
        nb_pred = nb.predict(x_test)
        
        # Random Forest
        rf = RandomForestClassifier(random_state=42)
        rf.fit(x_train, y_train)
        rf_pred = rf.predict(x_test)
        #collecting f1 score for each model 
        f1_2.append(f1_score(y_test, perc_pred, average='macro'))
        f1_3.append(f1_score(y_test, nb_pred, average='macro'))
        f1_4.append(f1_score(y_test, rf_pred, average='macro'))
        f1_all.append(sum(f1)/len(labels)) #macro f1 score for adaboost 
    return f1_2, f1_3, f1_4, f1_all

f1_p, f1_nb, f1_rf, f1_ada = get_f1() #f1 scores used for t test

def get_pvals (f1_p, f1_nb, f1_rf, f1_ada) :
    #running ttest and getting pvalues for each baseline in comparison to adaboost
    pval_p = ttest_rel(f1_ada, f1_p).pvalue
    pval_nb = ttest_rel(f1_ada, f1_nb).pvalue
    pval_rf = ttest_rel(f1_ada, f1_rf).pvalue
    return pval_p, pval_nb, pval_rf
pvals = get_pvals(f1_p, f1_nb, f1_rf, f1_ada)

Round:  1 | weighted error:  0.23986666666669426 | alpha:  0.5767053213319996
Round:  2 | weighted error:  0.24380371134980447 | alpha:  0.5659687716405064
Round:  3 | weighted error:  0.3351646758082875 | alpha:  0.3424586981378567
Round:  4 | weighted error:  0.3264214935073291 | alpha:  0.3622075412854935
Round:  5 | weighted error:  0.34973956371089393 | alpha:  0.31009208987637304
Round:  6 | weighted error:  0.4124247598307816 | alpha:  0.17697525861069907
Round:  7 | weighted error:  0.42501362982524116 | alpha:  0.15111254882500158
Round:  8 | weighted error:  0.3869033079287826 | alpha:  0.23017392379105325
Round:  9 | weighted error:  0.46612971650473006 | alpha:  0.06784446872291999
Round:  10 | weighted error:  0.43738203506007456 | alpha:  0.12589689601002113
Round:  11 | weighted error:  0.43317880904867034 | alpha:  0.13444664860140457
Round:  12 | weighted error:  0.3982906497851641 | alpha:  0.20629625345596975
Round:  13 | weighted error:  0.45716565271595905 | alpha:

In [12]:
#simple block for presentation purposes
print("\n\nF1 Scores: " , "\nPerceptron: ", f1_p, "\nNaive Bayes:", f1_nb, "\nRandom Forest: ", f1_rf, "\nAdaboost: ", f1_ada)
print("\n\nP-values: " "\nPerceptron: ", pvals[0], "|  Naive Bayes: ", pvals[1], "|  Random Forest: ", pvals[2])



F1 Scores:  
Perceptron:  [0.5832481056936869, 0.6870476156151827, 0.6333474562461298, 0.6729301510404692, 0.5994968308823163] 
Naive Bayes: [0.758109532965329, 0.7545006956438304, 0.761916011351254, 0.7623646702802599, 0.7335304368317912] 
Random Forest:  [0.7665360519955888, 0.7750264785589633, 0.7629763919960918, 0.779355284040288, 0.7776786546100272] 
Adaboost:  [np.float64(0.7922588483403631), np.float64(0.7955170090037661), np.float64(0.7887044658123826), np.float64(0.8089777405581517), np.float64(0.7881490320764666)]


P-values: 
Perceptron:  0.0008936127432817822 |  Naive Bayes:  0.0010863877087420976 |  Random Forest:  0.0025083762860335377
